# Threshold Schemes and Polynomial Secret Sharing

## Threshold Schemes

A $(t,n)$ threshold scheme lets a set of $n$ participants share a secret $s$ such that:

- any subset of size $t$ can recover $s$
- any subset of size less than $t$ learns nothing about $s$

For threshold signatures, this gives distributed control of a signing key without placing the full secret in one location.

## Shamir Secret Sharing: Why the Polynomial Matters

Shamir secret sharing encodes the secret as the constant term of a random polynomial over a finite field.

Given threshold $t$, dealer chooses random coefficients $a_1,\dots,a_{t-1}$ and defines:

$$
f(x) = s + \sum_{i=1}^{t-1} a_i x^i
$$

so that:

$$
f(0) = s
$$

Each participant $P_i$ receives share $(i, f(i))$.

### Reconstruction

A set $S$ of at least $t$ participant indices can recover $s$ using Lagrange interpolation at zero:

$$
s = f(0) = \sum_{i \in S} \lambda_i f(i),
$$

with

$$
\lambda_i = \prod_{j \in S, j \neq i} \frac{0-j}{i-j}.
$$

Everything is computed in the field, so division means multiplication by inverses.

### Why fewer than $t$ shares fail

A degree $(t-1)$ polynomial is not uniquely determined by fewer than $t$ points.
With fewer than $t$ shares, many different polynomials fit the observed points, each with a different constant term, so the secret remains information-theoretically hidden.

## Additive Secret Sharing and Why FROST Uses It

Shamir sharing represents a secret as points on a polynomial where $f(0)=s$.

Additive secret sharing uses a different representation: each participant picks a
value, and the secret is their sum.

For a signing subset of size $\alpha$, participants choose values $s_i$ such that:

$$
s = \sum_{i=1}^{\alpha} s_i.
$$

This has a key operational advantage: it is naturally non-interactive for share
creation, because each participant can choose its own contribution locally.

### From additive shares to Shamir-style shares

A useful bridge in threshold protocols is that additive shares can be locally
converted into values that behave like Shamir shares for the same secret.

Given participant set $S$ and Lagrange coefficients $\lambda_i$ at $x=0$, values
of the form

$$
\lambda_i s_i
$$

act as Shamir-style shares of the same aggregate secret $s$.

This idea appears in FROST signing: participants generate nonce contributions
additively, then use local weighting (with Lagrange coefficients and binding terms)
so the resulting values combine consistently under threshold interpolation logic.

In short:

- additive sharing gives efficient local contribution generation
- Lagrange weighting aligns those contributions with threshold reconstruction rules
- FROST uses this combination to build joint nonce material without running a full
  DKG for every signature

## Verifiable Secret Sharing (Feldman VSS)

Shamir sharing alone assumes the dealer is honest. Feldman VSS adds public commitments so each participant can check that their share matches a single committed polynomial.

Dealer publishes commitment vector:

$$
\tilde{C} = (\phi_0,\phi_1,\dots,\phi_{t-1}),
$$

where

$$
\phi_0 = g^s, \quad \phi_j = g^{a_j}.
$$

Participant $P_i$ with private share $f(i)$ verifies:

$$
g^{f(i)} \stackrel{?}{=} \prod_{j=0}^{t-1} \phi_j^{i^j}.
$$

If this check fails, the share is inconsistent with the committed polynomial and the participant can issue a complaint.

### Distributed consistency requirement

In distributed settings, all participants must see the same commitment vector $\tilde{C}$. Implementations typically ensure this by:

- using a trusted bulletin board or commitment server
- or adding a round where participants compare received commitments

This same consistency idea is important in threshold-signature systems like FROST.

## Connection to FROST

FROST signing uses long-lived key shares created by threshold-sharing techniques, and relies on participant consistency checks similar to these VSS ideas.

The polynomial viewpoint is the bridge:

- key ownership is split by polynomial shares
- threshold recovery behavior is controlled by interpolation
- validity is reinforced by public commitments

This is the mathematical foundation behind secure threshold key control.